# Example 8: Maps, Trajectories, and Spatial Analysis

Visualize AIS positions on interactive maps, derive vessel trajectories,
and explore movement patterns with MovingPandas.

Neptune's `[geo]` extra provides three spatial bridges:

| Bridge | Output | Use case |
|---|---|---|
| `positions_to_geodataframe()` | GeoDataFrame with Points | Position maps, spatial joins |
| `tracks_to_geodataframe()` | GeoDataFrame with LineStrings | Track maps, corridor analysis |
| `tracks_to_movingpandas()` | TrajectoryCollection | Speed profiles, trajectory splitting |

Plus `viz.py` helpers for density heatmaps, event markers, and trip animation.

## Prerequisites

```bash
pip install neptune-ais[geo,sql,notebooks]
```

The `[geo]` extra brings geopandas, shapely, movingpandas, lonboard, and h3.
GeoPandas `.explore()` renders interactive folium maps directly in Jupyter.

**Data requirement:** This example downloads NOAA data (~50-200 MB).
If you already ran [Example 02](02_archival_ingest.ipynb), the cached data will be reused.

## Setup and Data Loading

In [ ]:
import polars as pl
from neptune_ais.api import Neptune
from neptune_ais.geometry.bridges import (
    positions_to_geodataframe,
    tracks_to_geodataframe,
    tracks_to_movingpandas,
)
from neptune_ais.viz import prepare_density, prepare_events

n = Neptune("2024-06-15", sources=["noaa"], cache_dir="/tmp/neptune_demo")
n.download()

## Pick Vessels to Visualize

NOAA data has thousands of vessels per day. Plotting them all would
produce an unreadable map. We select a handful of **active, moving vessels**
— ones with many position reports and significant distance traveled —
to keep the visualizations clear.

In [ ]:
all_positions = n.positions().collect()
print(f"{len(all_positions):,} total positions from {all_positions['mmsi'].n_unique():,} vessels")

# Find vessels with the most position reports (active, likely moving)
vessel_counts = (
    all_positions
    .group_by("mmsi")
    .agg(
        reports=pl.len(),
        lat_range=pl.col("lat").max() - pl.col("lat").min(),
        lon_range=pl.col("lon").max() - pl.col("lon").min(),
    )
    # Filter for vessels that actually moved (not anchored)
    .filter((pl.col("lat_range") > 0.1) | (pl.col("lon_range") > 0.1))
    .sort("reports", descending=True)
)

# Pick the top 5 most active moving vessels
target_mmsis = vessel_counts.head(5)["mmsi"].to_list()
print(f"\nSelected {len(target_mmsis)} vessels: {target_mmsis}")
print(vessel_counts.head(5))

In [ ]:
# Filter positions to selected vessels
positions = all_positions.filter(pl.col("mmsi").is_in(target_mmsis))
print(f"{len(positions):,} positions for {len(target_mmsis)} vessels")

## 1. Positions on an Interactive Map

Convert positions to a GeoDataFrame with Point geometry, then use
`.explore()` for an interactive folium map. Each point is colored by vessel.

In [ ]:
gdf_positions = positions_to_geodataframe(positions)
print(f"GeoDataFrame: {gdf_positions.shape}, CRS: {gdf_positions.crs}")
gdf_positions.head(3)

In [ ]:
gdf_positions.explore(
    column="mmsi",
    cmap="Set1",
    marker_kwds={"radius": 2},
    style_kwds={"weight": 1},
    tooltip=["mmsi", "vessel_name", "timestamp", "sog"],
    legend=True,
)

## 2. Position Density Heatmap

`prepare_density()` bins **all** positions into H3 hexagonal cells and
counts observations per cell. This is useful for identifying high-traffic
areas without plotting millions of individual points.

Here we use the full dataset (not filtered to our 5 vessels) to show
overall maritime traffic patterns.

In [ ]:
density = prepare_density(all_positions, resolution=4)
print(f"{len(density)} H3 cells")
density.head(5)

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

# Convert density grid to GeoDataFrame for mapping
density_pd = density.to_pandas()
geometry = [Point(lon, lat) for lon, lat in zip(density_pd["center_lon"], density_pd["center_lat"])]
gdf_density = gpd.GeoDataFrame(density_pd, geometry=geometry, crs="EPSG:4326")

gdf_density.explore(
    column="count",
    cmap="YlOrRd",
    marker_kwds={"radius": 8},
    tooltip=["h3_index", "count"],
    legend=True,
)

## 3. Derive and Map Vessel Tracks

Neptune segments positions into tracks (voyages) by detecting time gaps.
With `as_geo=True`, the result is a GeoDataFrame with LineString geometry
ready for mapping.

We use `include_geometry=True` to generate actual WKB linestrings
(not just bounding boxes) — needed for accurate track rendering.

In [ ]:
# Create a vessel-filtered Neptune instance for track derivation
n_filtered = Neptune(
    "2024-06-15",
    sources=["noaa"],
    mmsi=target_mmsis,
    cache_dir="/tmp/neptune_demo",
)

gdf_tracks = n_filtered.tracks(
    include_geometry=True,
    as_geo=True,
    refresh=True,
)
print(f"{len(gdf_tracks)} tracks derived")
gdf_tracks[["track_id", "mmsi", "point_count", "distance_m", "duration_s", "mean_speed", "geometry"]].head()

In [ ]:
gdf_tracks.explore(
    column="mmsi",
    cmap="Set1",
    style_kwds={"weight": 3, "opacity": 0.8},
    tooltip=["mmsi", "point_count", "distance_m", "mean_speed", "duration_s"],
    legend=True,
)

## 4. MovingPandas Trajectory Analysis

MovingPandas adds trajectory-aware operations on top of GeoDataFrame:
speed profiles along a track, trajectory splitting, generalization,
and intersection detection.

`n.tracks(as_movingpandas=True)` returns a `TrajectoryCollection`
with one trajectory per vessel.

In [ ]:
tc = n_filtered.tracks(
    as_movingpandas=True,
    refresh=True,
)
print(f"{len(tc.trajectories)} trajectories")
for traj in tc.trajectories:
    print(f"  MMSI {traj.id}: {traj.get_length():.0f}m, "
          f"{traj.get_duration()}, "
          f"{len(traj.df)} points")

### Plot Trajectories Colored by Speed

MovingPandas can compute speed at each point along the trajectory.
Coloring by speed reveals navigation patterns: acceleration, cruising,
and deceleration zones.

In [ ]:
# Add speed column to each trajectory
tc = tc.add_speed(overwrite=True)

# Plot the first trajectory colored by speed
traj = tc.trajectories[0]
print(f"Plotting MMSI {traj.id}: {len(traj.df)} points")
traj.explore(
    column="speed",
    cmap="RdYlGn_r",
    style_kwds={"weight": 4, "opacity": 0.8},
    tooltip=["mmsi", "speed", "sog"],
    legend=True,
)

### Split Trajectories at Stops

MovingPandas can detect stops (periods where the vessel is stationary)
and split a single trajectory into sub-trips. This is useful for
identifying port calls and anchorages.

In [ ]:
from datetime import timedelta

# Split at stops: vessel stationary for >30min within 500m
split = tc.trajectories[0].split_by_observation_gap(timedelta(minutes=30))
print(f"Original trajectory split into {len(split)} segments")
for i, seg in enumerate(split.trajectories):
    print(f"  Segment {i}: {len(seg.df)} points, {seg.get_length():.0f}m, {seg.get_duration()}")

### Generalize Trajectories

Douglas-Peucker generalization reduces the number of points while
preserving the overall shape. Useful for rendering large tracks
without overwhelming the browser.

In [ ]:
traj = tc.trajectories[0]
generalized = traj.generalize(mode="dp", tolerance=0.001)  # ~100m tolerance in degrees
print(f"Original: {len(traj.df)} points → Generalized: {len(generalized.df)} points")
print(f"Reduction: {1 - len(generalized.df)/len(traj.df):.0%}")

## 5. All Trajectories on One Map

Plot all selected vessels' trajectories together using the
TrajectoryCollection's built-in `.explore()` method.

In [ ]:
tc.explore(
    column="mmsi",
    cmap="Set1",
    style_kwds={"weight": 3, "opacity": 0.8},
    tooltip=["mmsi", "speed"],
    legend=True,
)

## 6. Event Markers on a Map

If events have been derived (from [Example 04](04_event_detection.ipynb)),
`prepare_events()` clips and samples them for map rendering.
Each event becomes a point marker at its representative lat/lon.

In [ ]:
events_df = prepare_events(
    n.events(),
    max_events=200,
)
print(f"{len(events_df)} events")

if len(events_df) > 0:
    gdf_events = positions_to_geodataframe(events_df)
    gdf_events.explore(
        column="event_type",
        cmap="Set2",
        marker_kwds={"radius": 6},
        tooltip=["event_type", "mmsi", "start_time", "confidence_score"],
        legend=True,
    )
else:
    print("No events found. Run Example 04 first to derive events from positions.")

## Next Steps

- **[Example 02 — Archival Ingest](02_archival_ingest.ipynb)**: Download more data to visualize
- **[Example 03 — Multi-Source Fusion](03_multi_source_fusion.ipynb)**: Fuse NOAA + DMA before mapping
- **[Example 04 — Event Detection](04_event_detection.ipynb)**: Derive events to plot as markers
- **[Example 07 — Fishing Intelligence](07_fishing_intelligence.ipynb)**: GFW fishing effort grids